# 조치 없는 컨베이어 경고 검토 — 현장이 놓쳤나, 모델이 민감했나

정리본: [`경고_검토.md`](경고_검토.md). 대상: 컨베이어 경고 127건 중 사건 검토(C01~C05)에서 라벨이 붙지 않은 110건 (10-26 ~ 11-11).

**기준은 C03(TP)에서 확신을 준 네 가지다.**

| C03 W0069가 TP였던 근거 | 여기서 보는 것 |
|---|---|
| 지속 — 같은 센서·대차에서 며칠씩 같은 모양 | 엣지 가동일 기록(하루 칸)에서 그 칸이 이상이었던 날 수 |
| 재현 — DB로 다시 돌려도 같은 값 | 같은 시간 DB 재생 판정에 그 칸이 뜨나 |
| 크기 — 문턱을 넉넉히 넘음 | 하루 칸의 편차·재감지 크기 |
| 물리적 설명 | 재가동·측정 차이 같은 다른 설명이 없나 |

순서는 설계 문서 5-2를 따른다. ① 데이터 문제(재가동 과도·측정 차이)인가 → ② 피처가 날마다의 흔들림 넘게 실제로 벗어났나 (아니면 `모델과민`) → ③ 뚜렷하게 오래 벗어났나 (그러면 `현장놓침의심` → `FIELD_CHECK`).

| 절 | 덩어리 | 건수 |
|---|---|---|
| 2 | 대차 칸 (켜짐 길이·재감지) | 74 |
| 3 | 구동부 속도 | 22 |
| 4 | 유닛 지표 (+ C03에서 보류한 md_fwd 3건) | 14 + 3 |
| 5 | 결과 | |

In [1]:
import sys, json; sys.path.insert(0, "..")
from contextlib import redirect_stdout
from io import StringIO
import pandas as pd, numpy as np
from review_lib import paths, db, backend
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 40)
RC = paths.CACHE_DIR / "replay"
backend.use("v2.2")
with redirect_stdout(StringIO()):
    from tools_yeji.conveyor_models.core_judge import judge
    from tools_yeji.conveyor_models import core_realtime as cr
    from tools_yeji.conveyor_models.unit_params import UNIT_REF
TAGS = db.model_tags("conveyor", "v2.2")
EP = pd.read_csv(paths.EPISODES_DIR / "warning_episodes.csv", low_memory=False)
EV = pd.read_csv(paths.EPISODES_DIR / "warning_events.csv", low_memory=False, parse_dates=["ts"])
LB = pd.read_csv(paths.CHECK_DIRS["conveyor"] / "labels_episodes.csv", dtype=str, keep_default_na=False)
cv = EP[EP.domain == "conveyor"].merge(LB[["episode_id", "cm"]], on="episode_id")
todo = cv[cv.cm == ""].copy()
todo["덩어리"] = np.where(todo.key.str.contains(":wagons"), "대차 칸", np.where(todo.key.str.contains("drive"), "구동부", "유닛"))
print("컨베이어 경고", len(cv), "· 라벨 없음", len(todo)); todo.groupby("덩어리").size()

컨베이어 경고 127 · 라벨 없음 110


덩어리
구동부     22
대차 칸    74
유닛      14
dtype: int64

## 1. 증거 자료

| 자료 | 기간 | 뜻 |
|---|---|---|
| 엣지 가동일 기록 `conveyor_core_history.json` | 09-10 ~ 11-05 (10-26부터 엣지 실시간) | 엣지가 직접 남긴 하루 칸. 엣지 원시 입력을 볼 수 없으니 이것이 "엣지가 본 것"의 가장 가까운 증거 |
| DB 재생 `c02_edge_1026_1105` | 10-26 02:15 ~ 11-05 | 엣지 조건 그대로(씨앗 기록) 운영 코어 재생. 15분마다 판정 |
| DB 재생 `fp_late_1105_1111` | 11-05 ~ 11-11 09:00 | 이번에 새로 돌림. 11-05 이후는 엣지 가동일 기록이 없어 이 재생의 하루 칸(DB)으로 대신 |
| 유닛 사이클 | 09-10 ~ 11-11 | 위 재생과 C01 노트북 캐시의 유닛 사이클 (DB) |

In [2]:
R1 = pd.read_pickle(RC / "c02_edge_1026_1105.pkl")
f = RC / "fp_late_1105_1111.pkl"
if not f.exists():   # 처음 한 번 약 8분. 11-06 ~ 11-11 원시값은 db 캐시에서 (없으면 DB에서 하루씩)
    HIST = json.load(open(paths.MODEL_VERSIONS["v2.2"]["backend"] / "data" / "conveyor_core_history.json"))
    from review_lib import replay
    raw = db.fetch_range("2026-11-05 00:00", "2026-11-11 09:00", TAGS, group="conveyor")
    P, E, core = replay.conveyor_core(raw, history=[r for r in HIST if r["day"] < "2026-11-05"], units=("md_fwd", "md_rev", "airbag"))
    pd.to_pickle({"passes": P, "evals": E, "cycles": core.unit_cycles, "hz": raw[["Hz_In_Conv"]].rename(columns={"Hz_In_Conv": "Hz"}),
                  "history": [{**r, "day": str(pd.Timestamp(r["day"]).date())} for r in core.history]}, f)
R2 = pd.read_pickle(f)
H = pd.DataFrame(json.load(open(paths.MODEL_VERSIONS["v2.2"]["backend"] / "data" / "conveyor_core_history.json")))
Hx = pd.DataFrame(R2["history"]); Hx = Hx[Hx.day > H.day.max()]
H = pd.concat([H, Hx], ignore_index=True); H["day"] = pd.to_datetime(H.day)
J = judge(H)
EVW = pd.concat([R1["evals"], R2["evals"]]); EVW = EVW[EVW.wagon.notna()].copy(); EVW["t"] = pd.to_datetime(EVW.t)
print("하루 칸:", J.day.min().date(), "~", J.day.max().date(), "· 재생 판정:", EVW.t.min(), "~", EVW.t.max())

하루 칸: 2026-09-10 ~ 2026-11-10 · 재생 판정: 2026-10-26 07:23:20.103134 ~ 2026-11-11 08:51:53.739821


## 2. 대차 칸 74건

경고마다:
- `하루칸` — 경고가 난 가동일에 그 센서·대차 하루 칸이 이상 상태였던 대차 수 / 경고 대차 수
- `재생` — 경고 시간(앞뒤 30분) DB 재생 판정에 그 대차가 뜬 수 / 경고 대차 수
- `켜짐길이_최대` · `앞끝_최소` · `재감지_최대` — 그날 하루 칸 값 (문턱: 켜짐 길이 +3, 재감지 0.5)

In [3]:
REASON_STATE = {"wagon_rearm": "이물(분리)", "wagon_debris_front": "이물(앞)", "wagon_debris_back": "이물(뒤)",
                "wagon_debris": "이물(기타)", "wagon_shift": "위치 변화", "wagon_jitter": "흔들림"}
bad = lambda s: s.str.startswith("이물") | s.isin(["위치 변화", "흔들림"])
rows = []
for r in todo[todo.덩어리 == "대차 칸"].itertuples():
    s = r.key.split(":")[0]; ws = [int(x) for x in str(r.wagon_nos).split(";") if x and x != "nan"]
    a = pd.Timestamp(r.onset); b = pd.Timestamp(r.last_detected) if isinstance(r.last_detected, str) else a
    d0, d1 = (a - pd.Timedelta(hours=6)).normalize(), (b - pd.Timedelta(hours=6)).normalize()
    win = J[(J.sensor == s) & J.wagon.isin(ws) & J.day.between(d0, d1)]
    e = EVW[(EVW.sensor == s) & EVW.wagon.isin(ws) & EVW.t.between(a - pd.Timedelta("30min"), b + pd.Timedelta("30min"))]
    real = sorted(set(win[bad(win.state)].wagon.astype(int)) | set(e.wagon.astype(int)))
    rows.append(dict(episode_id=r.episode_id, 센서=s, 대차=";".join(map(str, ws)),
                     상태="/".join(sorted({REASON_STATE.get(x, x) for x in str(r.reasons).split(";")})),
                     시작=f"{a:%m-%d %H:%M}", 시간_h=round(r.duration_h, 1),
                     하루칸=f"{win[bad(win.state)].wagon.nunique()}/{len(ws)}", 재생=f"{e.wagon.nunique()}/{len(ws)}",
                     켜짐길이_최대=win.width_ex.max(), 앞끝_최소=win.rise_off.min(), 재감지_최대=round(win.rearm.max(), 2),
                     실제대차=";".join(map(str, real))))
W = pd.DataFrame(rows)
W

,episode_id,센서,대차,상태,시작,시간_h,하루칸,재생,켜짐길이_최대,앞끝_최소,재감지_최대,실제대차
0,W0070,P00164,20,이물(분리)/이물(앞),10-26 07:06,267.7,1/1,1/1,5.00,-7.0,0.77,20
1,W0071,P00140,6;19;28;32;37,이물(뒤),10-26 07:07,6.0,0/5,0/5,1.00,0.0,0.00,
2,W0075,P00154,4;6;10;19;24;31,이물(뒤),10-26 07:19,6.6,0/6,0/6,1.25,-1.0,0.00,
3,W0078,P00141,26;35;36,이물(뒤),10-26 08:42,3.4,0/3,0/3,0.75,-1.0,0.00,
4,W0084,P00141,35,이물(뒤),10-26 17:12,4.9,0/1,0/1,0.75,0.0,0.00,
5,W0086,P00141,29;35,이물(뒤),10-27 00:20,13.1,0/2,0/2,1.00,0.0,0.00,
6,W0092,P00154,9;32,이물(분리)/이물(앞),10-27 07:11,6.9,0/2,1/2,0.00,0.0,0.37,9
7,W0095,P00140,15,이물(뒤),10-27 09:09,0.2,0/1,0/1,1.00,0.0,0.00,
8,W0101,P00140,12,이물(뒤),10-27 23:23,7.5,0/1,0/1,1.00,0.0,0.00,
9,W0108,P00140,16;24,이물(뒤),10-28 13:43,3.3,0/2,0/2,0.00,0.0,0.00,


**74건 중 63건은 하루 칸에도, DB 재생에도 흔적이 없다**(대부분 표식 뒤쪽 이물, 4건은 앞쪽 이물 포함). 경고가 난 날 엣지 자신의 하루 칸에서 그 대차의 켜짐 길이 편차는 최대 +1.2(문턱 +3)이고,
재감지도 0이다. 대부분 몇 시간짜리다. C03과 정반대 모양이다 → `모델과민`. C02의 W0098과 같은 종류이고, 모델 수정 예정 C-04(켜짐 길이 1카운트 흔들림)의 근거가 늘었다.

흔적이 있는 것은 대차 셋이다. 하루 칸 전체 이력으로 얼마나 오래된 것인지 본다.

In [4]:
real_cells = sorted({(r.센서, int(w)) for r in W.itertuples() for w in r.실제대차.split(";") if w})
out = []
for s, w in real_cells:
    g = J[(J.sensor == s) & (J.wagon == w)].sort_values("day")
    out.append(dict(센서=s, 대차=w, 이상일=f"{bad(g.state).sum()}/{len(g)}", 재감지_중앙값=round(g.rearm.median(), 2),
                    최근5일_재감지=" ".join(f"{x:.2f}" for x in g.rearm.tail(5)), 최근5일_앞끝=" ".join(f"{x:.0f}" for x in g.rise_off.tail(5)),
                    첫날_상태=g.state.iloc[0], 경고수=int(W.실제대차.str.split(";").apply(lambda L: str(w) in L).where(W.센서 == s, False).sum())))
CELLS = pd.DataFrame(out)
CELLS

,센서,대차,이상일,재감지_중앙값,최근5일_재감지,최근5일_앞끝,첫날_상태,경고수
0,P00154,9,0/44,0.00,0.00 0.00 0.00 0.00 0.00,0 0 0 0 0,정상,1
1,P00164,20,44/44,0.60,0.36 0.67 0.56 0.83 0.90,-6 -7 -6 -6 -6,이물(분리),2
2,P00459,9,20/44,0.23,0.27 0.88 0.56 0.79 0.73,0 -14 -12 -14 -13,이물(분리),8


- **P00164(오프닝 진입) 대차 20 — 하루 칸 44일 전부 이상.** 매일 표식 6~7카운트 앞에서 켜지고 켜짐 길이 +5~6, 재감지 0.24~0.88.
  C03과 같은 "대차 구조물을 센서가 보는" 모양이 9-10부터 이어졌는데 일지에 기록이 없다 → **`현장놓침의심` → `FIELD_CHECK`**.
- **P00459(후진 진입) 대차 9 — 하루 칸 44일 중 20일 이상, 11월 들어 재감지 0.56~0.88로 늘었다**(표식 12~14카운트 앞). → **`FIELD_CHECK`**.
  같은 경고에 묶인 다른 대차(41·15·27·42·24)는 흔적이 없다.
- **P00154(클로징 진입) 대차 9 — 10-27 하루만 재감지(0.37), DB 재생 07:17~13:32 재현.** 앞뒤 날은 0이다. 무언가 붙었다 떨어진 것으로 보인다 → `HOLD`.
- 셋 다 1번 대차가 아니라 PLC 번호 이상으로 이어지지는 않는다(C03).

## 3. 구동부 속도 22건

경고가 켜진 순간마다 운영 코어를 그 직전 2.5시간부터 다시 흘려, 속도 창 표본 수·편차와 직전 재가동을 본다(C05 3절과 같은 방법).
처음 한 번 약 3분, 결과는 캐시한다.

In [5]:
f = RC / "fp_drive_check.pkl"
if f.exists():
    DRV = pd.read_pickle(f)
else:
    days = {}
    def raw_between(a, b):
        parts = []
        for x in pd.date_range(a.normalize(), b.normalize()):
            if x not in days: days[x] = db.fetch_day(x, TAGS, group="conveyor")
            parts.append(days[x])
        r = pd.concat(parts); return r[(r.index >= a) & (r.index <= b)]
    drv = cv[cv.key.str.contains("drive")]
    d = EV[(EV.kind == "conveyor") & EV.key.astype(str).str.contains("drive")].sort_values("ts")
    rows = []
    for r in drv.itertuples():
        a, b = pd.Timestamp(r.onset), pd.Timestamp(r.last_detected)
        for t in d[d.ts.between(a - pd.Timedelta("1s"), b + pd.Timedelta("1s")) & (d.active.astype(str) == "True")].ts:
            raw = raw_between(t - pd.Timedelta("150min"), t + pd.Timedelta("5min"))
            core = cr.CoreRealtime(history_path=None)
            cols = [c for c in raw.columns if c != "source"]; vals = raw[cols].to_numpy(dtype=object)
            run = raw.Conv_Run_Out.ffill().fillna(0); ups = raw.index[run.diff() == 1]; prev = ups[ups <= t + pd.Timedelta("30s")]
            n_at = dev_at = None; cross = []
            for tt, v in zip(raw.index, vals):
                core.update(tt, {c: x for c, x in zip(cols, v) if x is not None and x == x})
                if tt < t - pd.Timedelta("5min") or not core.speed: continue
                arr = np.array([(x[1], x[2]) for x in core.speed]); n = len(arr)
                dev = (arr[:, 0].sum() / arr[:, 1].sum() / cr.K_SPEED - 1) * 100
                if n_at is None and tt >= t: n_at, dev_at = n, dev
                if n >= 100 and abs(dev) > 0.3: cross.append(tt)
            near = [x for x in cross if abs((x - t).total_seconds()) <= 300]
            rows.append(dict(episode_id=r.episode_id, 켜짐=t, 직전재가동_분=round((t - prev[-1]).total_seconds() / 60, 1) if len(prev) else None,
                             표본=n_at, 편차_pct=round(dev_at, 3) if dev_at is not None else None, DB재현=near[0] if near else pd.NaT))
    DRV = pd.DataFrame(rows); DRV.to_pickle(f)
DRV

,episode_id,켜짐,직전재가동_분,표본,편차_pct,DB재현
0,W0064,2026-10-26 06:19:05.749,80.1,NaN,NaN,NaT
1,W0080,2026-10-26 11:27:39.637,1.2,154.0,-0.100,NaT
2,W0087,2026-10-27 06:21:00.814,3.9,176.0,0.144,NaT
3,W0112,2026-10-28 19:51:05.606,4.8,857.0,0.065,NaT
4,W0118,2026-10-29 00:36:25.388,10.2,395.0,-0.069,NaT
5,W0118,2026-10-29 06:21:55.954,2.0,140.0,0.138,2026-10-29 06:21:37.921910
6,W0131,2026-10-29 11:27:16.766,1.8,179.0,-0.086,NaT
7,W0131,2026-10-29 11:35:17.216,9.8,717.0,-0.056,NaT
8,W0145,2026-10-30 11:30:26.223,2.5,170.0,-0.403,2026-10-30 11:29:17.163619
9,W0154,2026-10-30 19:52:59.660,4.0,305.0,-0.030,NaT


- 켜진 32번이 **모두 재가동 뒤 25분 안**이다(10-26 06:19는 엣지 모델이 처음 켜진 순간). 대부분 속도 창 표본이 90~440개뿐일 때다. 11번은 DB 재생도 같은 몇 분 안에 문턱을 넘는다.
- 정속 구간 속도 비율은 10-29 ~ 11-05 매일 0.2743 ~ 0.2745(기준 0.2744)다(C04·C05 노트북 4절). 구동부는 멀쩡하다.
- → 22건 모두 **`재가동과도`**. 모델 수정 예정 C-06.

## 4. 유닛 지표 17건 (C03에서 보류한 md_fwd 주입 시간 3건 포함)

유닛 판정은 오늘 사이클이 2,000개 넘으면 "오늘 지금까지의 평균", 그 전에는 "마지막 가동일 기록"을 고정 기준(`UNIT_REF`)과 비교한다. 5σ 넘게 느리면 점검 추천.

### 4-1. 엣지가 잰 유닛 시간 지표는 DB보다 일정하게 길다

같은 날, 엣지 가동일 기록(10-26부터 엣지 실시간)과 DB 재생 사이클의 하루 평균. 10-19까지 기록은 DB 배치 씨앗이다.

In [6]:
C = pd.concat([pd.read_pickle(RC / f"c01_units_{a}.pkl")["cycles"] for a in ["2026-09-10_2026-09-24", "2026-10-01_2026-10-15", "2026-10-16_2026-10-27"]]
              + [R1["cycles"], R2["cycles"]])
C["t"] = pd.to_datetime(C.t); C = C.drop_duplicates(["unit", "t"]).sort_values("t"); C["day"] = (C.t - pd.Timedelta(hours=6)).dt.strftime("%Y-%m-%d")
M = {"md_fwd": "air_time", "md_rev": "stop_rev_time", "airbag": "nozzle_rev_time_1"}
cmp = {}
for u, m in M.items():
    UH = pd.DataFrame(json.load(open(paths.MODEL_VERSIONS["v2.2"]["backend"] / "data" / f"conveyor_unit_{u}_history.json"))).set_index("day")[m]
    dbm = C[C.unit == u].groupby("day")[m].mean()
    days = [d for d in UH.index if d >= "2026-10-15"]
    cmp[(u, m, "엣지 기록")] = UH.loc[days]; cmp[(u, m, "DB")] = dbm.reindex(days)
CMP = pd.DataFrame(cmp).T; CMP.columns = [c[5:] for c in CMP.columns]
CMP.round(4)

10-15   10-16   10-17   10-19   10-26   10-27   10-28   10-29   10-30   10-31   11-02   11-03   11-04   11-05
md_fwd air_time          엣지 기록  0.6377  0.6356  0.6273  0.6233  0.6762  0.6864  0.6703  0.6664  0.6947  0.6680  0.6853  0.6736  0.6776  0.6618
                         DB     0.6373  0.6354  0.6277  0.6229  0.6262  0.6377  0.6265  0.6219  0.6470  0.6240  0.6395  0.6280  0.6307  0.6238
md_rev stop_rev_time     엣지 기록  0.3558  0.3536  0.3555  0.3529  0.3602  0.3631  0.3659  0.3654  0.3672  0.3656  0.3635  0.3636  0.3640  0.3736
                         DB     0.3554  0.3533  0.3563  0.3532  0.3540  0.3602  0.3576  0.3624  0.3563  0.3587  0.3578  0.3557  0.3600  0.3626
airbag nozzle_rev_time_1 엣지 기록  0.2993  0.2967  0.2962  0.2876  0.2985  0.2949  0.2987  0.3060  0.3021  0.3090  0.3035  0.2999  0.3038  0.3108
                         DB     0.2993  0.2967  0.2962  0.2876  0.2923  0.2962  0.2908  0.2998  0.2967  0.2982  0.2936  0.2932  0.2982  0.2963

- md_fwd 주입 시간: DB 0.62 ~ 0.65초(10-19 전후 계단 없음), 엣지 0.66 ~ 0.69초. **엣지가 약 0.045초 길다.**
  md_rev 스톱퍼 후진 +0.006 ~ 0.011초, 에어백 노즐 후진 +0.005 ~ 0.012초도 같은 방향이다. 에어백 스톱퍼 후진은 둘이 같다.
- 판정 기준(`UNIT_REF`·씨앗)은 DB로 잡혔고 10-26부터 엣지 값으로 판정하니, DB에는 없는 "느려짐"이 생긴다.
  md_fwd 주입 시간 기준값은 0.899초로 실제(0.62)와도 동떨어져 있다 — "빨라짐" 3일 승인으로 기준이 0.62대로 옮겨 간 뒤, 엣지의 0.68을 느려짐으로 봤다.
- 엣지가 PLC를 읽는 주기·방식이 DB와 다른 것으로 보인다(C-04와 같은 질문).

### 4-2. 경고가 켜진 순간의 DB z

In [7]:
rows = []
U = cv[cv.sub_unit.isin(["md_fwd", "md_rev", "airbag"]) & ~cv.key.str.contains("wagons|drive|P00451")]
for r in U.itertuples():
    for reason in r.reasons.split(";"):
        tail = reason.split("_")[-1]; metric = reason[: -len(tail) - 1]
        ref, sig = UNIT_REF[r.sub_unit].get(metric, (np.nan, np.nan))
        t = pd.Timestamp(r.onset); d0 = (t - pd.Timedelta(hours=6)).normalize() + pd.Timedelta(hours=6)
        c = C[(C.unit == r.sub_unit) & (C.t >= d0) & (C.t < d0 + pd.Timedelta(days=1))]
        prev = C[(C.unit == r.sub_unit) & (C.t < d0)]; pday = prev.day.iloc[-1] if len(prev) else None
        now = c[c.t <= t]
        basis = now if len(now) >= 2000 else prev[prev.day == pday]          # 엣지와 같은 판정 근거 (오늘 2,000개 전에는 마지막 가동일)
        v = basis[metric].mean()
        rows.append(dict(episode_id=r.episode_id, 유닛=r.sub_unit, 지표=metric, 종류=tail, 시작=f"{t:%m-%d %H:%M}", 시간_h=round(r.duration_h, 1),
                         오늘사이클=len(now), 근거="오늘" if len(now) >= 2000 else f"전날({pday[5:] if pday else '-'})",
                         기준=ref, DB값=round(v, 4), DB_z=round((v - ref) / sig, 1), cm=r.cm))
UZ = pd.DataFrame(rows)
UZ

,episode_id,유닛,지표,종류,시작,시간_h,오늘사이클,근거,기준,DB값,DB_z,cm
0,W0081,md_fwd,air_time,slow,10-26 14:38,48.0,2065,오늘,0.8989,0.6218,-38.5,HOLD
1,W0082,md_rev,stop_fwd_time,slow,10-26 14:38,242.9,2083,오늘,0.2795,0.2887,2.7,
2,W0082,md_rev,stop_rev_time,slow,10-26 14:38,242.9,2083,오늘,0.3445,0.3555,3.0,
3,W0114,md_fwd,air_time,slow,10-28 20:15,21.8,3352,오늘,0.8989,0.6240,-38.2,HOLD
4,W0134,airbag,stop_rev_time,slow,10-29 15:40,2.8,2069,오늘,0.3178,0.3377,4.6,
5,W0148,md_fwd,air_time,slow,10-30 14:42,167.8,2005,오늘,0.8989,0.6378,-36.3,HOLD
6,W0168,airbag,nozzle_rev_time_1,slow,10-31 15:42,6.0,2009,오늘,0.2895,0.3026,3.1,
7,W0206,airbag,stop_rev_time,slow,11-03 18:47,0.4,2482,오늘,0.3178,0.3362,4.3,
8,W0262,airbag,nozzle_rev_time_1,slow,11-06 15:16,0.8,2001,오늘,0.2895,0.3031,3.2,
9,W0265,md_rev,stop_rev_time,slow,11-06 17:25,93.1,2835,오늘,0.3445,0.3617,4.7,


- **오늘 사이클이 막 2,000개를 넘긴 순간(2,001 ~ 2,224개)에 켜진 경고가 대부분이다.** 판정 근거가 "어제 기록"에서 "오늘 평균"으로 바뀌는 순간이고, 이때 오늘 평균에는 아침 시작 구간이 많이 섞여 높게 나온다. 하루가 쌓이면 내려가 꺼진다.
- 그 순간 **DB z는 2.7 ~ 4.7로 문턱(5) 아래**다. 엣지 측정 차이(4-1)만큼 더해져 넘었다. md_fwd 주입 시간은 4-1의 기준 이동 때문이다.
- W0282(md_rev 걸림 거리): 전날(11-06) DB z −4.9. σ 하한 0.05카운트가 이 지표의 평소 날 흔들림(±0.1)보다 작다.
- W0283(에어백 노즐 후진, 57시간): 전날(11-06) DB z 2.8. 11-07·08은 짧은 날(1,000사이클 미만)이라 기록이 안 남아 11-06 기록이 주말 내내 판정 근거로 남았다.
- → 위는 모두 **`모델과민`**(기준·측정 차이, 2,000사이클 전환). 모델 수정 예정 C-07.

### 4-3. md_rev 스톱퍼는 9월 중순에 실제로 느려졌다 (W0082 · W0265 · W0327)

In [8]:
r = C[C.unit == "md_rev"].groupby("day").agg(n=("t", "size"), 스톱퍼전진=("stop_fwd_time", "mean"), 스톱퍼후진=("stop_rev_time", "mean"))
r = r[r.n > 1000]; r.index = r.index.str[5:]
print("기준: 전진", UNIT_REF["md_rev"]["stop_fwd_time"], "· 후진", UNIT_REF["md_rev"]["stop_rev_time"], "(값, σ) — 처음 3 가동일(09-10~12)")
r.T.round(4)

기준: 전진 (0.2795, 0.0034) · 후진 (0.3445, 0.0037) (값, σ) — 처음 3 가동일(09-10~12)


day,09-10,09-11,09-12,09-14,09-15,09-16,09-17,09-18,09-19,09-21,09-22,09-23,10-01,10-02,10-03,10-05,10-06,10-07,10-08,10-09,...,10-17,10-19,10-20,10-21,10-22,10-23,10-24,10-26,10-27,10-28,10-29,10-30,10-31,11-02,11-03,11-04,11-05,11-06,11-09,11-10
n,4454.0000,4471.0000,4475.0000,4289.0000,4401.0000,4335.0000,4333.0000,4404.0000,4385.0000,4466.0000,4413.0000,4524.0000,4410.0000,4403.0000,4378.0000,4380.0000,4422.0000,4498.0000,4557.0000,4545.0000,...,4392.0000,4551.0000,4472.0000,4456.0000,4479.0000,4357.0000,4476.0000,4516.0000,4550.0000,4375.0000,4420.0000,4297.0000,4436.0000,4413.0000,3933.0000,3733.0000,4593.0000,4495.0000,4466.0000,4311.0000
스톱퍼전진,0.2830,0.2792,0.2764,0.2758,0.2835,0.2839,0.2800,0.2794,0.2886,0.2922,0.2909,0.2821,0.2856,0.2827,0.2868,0.2939,0.2903,0.2848,0.2889,0.2944,...,0.2976,0.2916,0.2886,0.2992,0.2931,0.2938,0.2864,0.2918,0.2872,0.2986,0.2918,0.2917,0.2889,0.2976,0.2903,0.2966,0.2925,0.2975,0.3004,0.2920
스톱퍼후진,0.3457,0.3434,0.3453,0.3376,0.3566,0.3466,0.3498,0.3495,0.3560,0.3596,0.3556,0.3512,0.3550,0.3464,0.3553,0.3601,0.3516,0.3440,0.3550,0.3526,...,0.3563,0.3532,0.3541,0.3573,0.3607,0.3625,0.3544,0.3540,0.3602,0.3576,0.3624,0.3563,0.3587,0.3578,0.3557,0.3600,0.3626,0.3608,0.3630,0.3591


- DB로도 스톱퍼 전진 0.276 ~ 0.284 → 0.285 ~ 0.299초, 후진 0.343 ~ 0.346 → 0.350 ~ 0.364초로 **9-15 ~ 19 무렵 약 +0.01초 계단**이 있고, 그 뒤 두 달 그대로다.
  기준은 계단 전 3일로 잡혔다. 점점 나빠지는 추세는 아니다.
- DB z는 2.6 ~ 4.7로 문턱 아래, 엣지 측정 차이가 더해져 넘었다. 실제 변화(작고 오래됨)와 측정 차이가 섞였다 → `HOLD`. 9월 중순에 중형 후진 스톱퍼 쪽 공압·속도 조절을 손댔는지 현장에 묻는다.

## 5. 결과

In [9]:
lab = []
fc = {("P00164", 20): "2라인 캐리어 20번 오프닝 진입 센서(P00164) 표식 6~7카운트 앞 구조물 — 9-10부터 매일 그 자리에서 먼저 켜짐",
      ("P00459", 9): "2라인 캐리어 9번 후진 진입 센서(P00459) 표식 12~14카운트 앞 구조물 — 간헐 재감지가 11월 들어 늘어남"}
for r in W.itertuples():
    rw = [int(x) for x in r.실제대차.split(";") if x]
    hit = [(r.센서, w) for w in rw if (r.센서, w) in fc]
    if hit:
        lab.append((r.episode_id, "FIELD_CHECK", "현장놓침의심", fc[hit[0]]))
    elif rw:
        lab.append((r.episode_id, "HOLD", "", ""))
    else:
        lab.append((r.episode_id, "FP", "모델과민", ""))
for e in DRV.episode_id.unique():
    lab.append((e, "FP", "재가동과도", ""))
for e in UZ.episode_id.unique():
    lab.append((e, "HOLD" if e in ("W0082", "W0265", "W0327") else "FP", "" if e in ("W0082", "W0265", "W0327") else "모델과민",
                "9월 중순 2라인 중형 후진 스톱퍼 공압·속도 조절 변경 이력" if e in ("W0082", "W0265", "W0327") else ""))
LAB = pd.DataFrame(lab, columns=["episode_id", "cm", "fp_cause", "field_check_req"]).drop_duplicates("episode_id")
LAB = LAB[LAB.episode_id.isin(set(todo.episode_id) | {"W0081", "W0114", "W0148"})]
LAB.merge(todo[["episode_id", "덩어리"]], on="episode_id", how="left").fillna({"덩어리": "유닛(C03 보류)"}).groupby(["덩어리", "cm", "fp_cause"]).size()

덩어리         cm           fp_cause
구동부         FP           재가동과도       22
대차 칸        FIELD_CHECK  현장놓침의심      10
            FP           모델과민        63
            HOLD                      1
유닛          FP           모델과민        11
            HOLD                      3
유닛(C03 보류)  FP           모델과민         3
dtype: int64

**결론**

1. 조치 없는 경고 110건(+ C03에서 보류한 3건) 가운데 **진짜인데 현장이 안 본 것은 대차 두 칸**이다. 경고로는 10건이다.
   - P00164 대차 20 (2건) — 9-10부터 매일, C03과 같은 모양
   - P00459 대차 9 (8건) — 간헐, 11월 들어 늘어남
   - 둘 다 `FIELD_CHECK`
2. **대부분은 모델이 민감했다.**
   - 대차 칸 63건(켜짐 길이·앞 끝): 하루 칸·DB 재생에 흔적 없음 → C-04
   - 구동부 22건: 재가동 직후 → C-06
   - 유닛 14건(C03 보류 3건 포함): 엣지·DB 측정 차이와 2,000사이클 전환 → C-07
3. 판단 보류는 4건이다. P00154 대차 9의 하루짜리 재감지(W0092)와 md_rev 스톱퍼 9월 계단(W0082 · W0265 · W0327).
4. C03 조치 뒤 잔여 경고 7건(P00451 4·9·10·13·22번 등)은 이미 `HOLD`다. 하루 칸·재생 모두 실제로 보이는 재감지다.